In [ ]:
# ============================================================
# FIXED PaDiM Essential Edge Metrics
# Method: PaDiM + Max Patch Score
# Dataset: Lusitano_Dataset
# SEED = 42 | No CenterCrop
#
# Fixes:
#   1. Forces CUDA GPU; stops if GPU is not active
#   2. Uses local /content/Lusitano_Dataset
#   3. Skips corrupt/unreadable images safely
#   4. Uses NUM_WORKERS = 0 to avoid DataLoader worker crash
#   5. Reports compute-only and end-to-end local runtime
# ============================================================

import os
import gc
import time
import random
import shutil
import psutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights
from PIL import Image, ImageFile, UnidentifiedImageError

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# 1) Reproducibility
# ============================================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

ImageFile.LOAD_TRUNCATED_IMAGES = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass

# ============================================================
# 2) Force GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not active. Go to Runtime > Change runtime type > GPU, "
        "then restart the runtime and run again."
    )

DEVICE = "cuda"
print("Using device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 3) Dataset paths: Drive -> local /content
# ============================================================

COPY_DATASET_TO_LOCAL = True

DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/<YOUR_DATASET_FOLDER>/Lusitano_Dataset")
LOCAL_DATASET_ROOT = Path("/content/Lusitano_Dataset")

if COPY_DATASET_TO_LOCAL:
    if not DRIVE_DATASET_ROOT.exists():
        raise ValueError(f"Drive dataset not found: {DRIVE_DATASET_ROOT}")

    expected_local_train = LOCAL_DATASET_ROOT / "nondefects" / "nondefects"
    expected_local_test = LOCAL_DATASET_ROOT / "test" / "test"

    if expected_local_train.exists() and expected_local_test.exists():
        print("Local dataset already exists:", LOCAL_DATASET_ROOT)
    else:
        if LOCAL_DATASET_ROOT.exists():
            print("Removing incomplete local dataset copy...")
            shutil.rmtree(LOCAL_DATASET_ROOT)

        print("Copying dataset from Google Drive to local Colab storage...")
        print("Source:", DRIVE_DATASET_ROOT)
        print("Target:", LOCAL_DATASET_ROOT)

        copy_start = time.time()
        shutil.copytree(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT)
        print(f"Dataset copy completed in {(time.time() - copy_start) / 60:.2f} minutes.")

    DATASET_ROOT = LOCAL_DATASET_ROOT
else:
    DATASET_ROOT = DRIVE_DATASET_ROOT

print("Using DATASET_ROOT:", DATASET_ROOT)

train_good_path = DATASET_ROOT / "nondefects" / "nondefects"
test_root_path  = DATASET_ROOT / "test" / "test"

if not train_good_path.exists():
    raise ValueError(f"Training path not found: {train_good_path}")

if not test_root_path.exists():
    raise ValueError(f"Test path not found: {test_root_path}")

print("Train folder:", train_good_path)
print("Test folder :", test_root_path)

# ============================================================
# 4) Experiment settings
# ============================================================

METHOD_NAME = "PaDiM + Max Patch Score"

IMG_SIZE = 448
BATCH_SIZE = 4

# Keep 0 for stability in Colab.
NUM_WORKERS = 0

FEAT_GRID = 28

# 600 works but is large. Use 544 for safer PaDiM memory/time.
# Set back to 600 only if you specifically want exact old setting.
EMB_DIM = 544

COV_EPS = 1e-4
THRESH_SAMPLE_IMAGES = 2000

SCORE_MODE = "max"
TOPK_FRAC = None

SAVE_DIR = Path("/content/drive/MyDrive/<YOUR_OUTPUT_FOLDER>/localcopy_compute_timing_fixed")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

RESULT_CSV = SAVE_DIR / "padim_max_patch_score_fixed_seed42.csv"
SCORES_CSV = SAVE_DIR / "padim_max_patch_score_fixed_image_scores_seed42.csv"
BAD_IMAGES_CSV = SAVE_DIR / "padim_skipped_bad_images_seed42.csv"

# ============================================================
# 5) Metric / memory helpers
# ============================================================

def bytes_to_mb(x):
    return x / (1024 ** 2)

def tensor_size_mb(tensor):
    return bytes_to_mb(tensor.numel() * tensor.element_size())

def model_size_mb(model):
    total_bytes = 0

    for p in model.parameters():
        total_bytes += p.numel() * p.element_size()

    for b in model.buffers():
        total_bytes += b.numel() * b.element_size()

    return bytes_to_mb(total_bytes)

def current_ram_mb():
    process = psutil.Process(os.getpid())
    return bytes_to_mb(process.memory_info().rss)

def reset_peak_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

def get_peak_memory_mb():
    torch.cuda.synchronize()
    return bytes_to_mb(torch.cuda.max_memory_allocated())

def edge_efficiency_score(auc, ap, f1, time_per_image, total_mb, peak_gpu_mb):
    numerator = 0.25 * auc + 0.35 * ap + 0.40 * f1
    denominator = 0.20 * time_per_image + 0.40 * total_mb + 0.40 * peak_gpu_mb

    if denominator <= 0:
        return 0.0

    return numerator / denominator

# ============================================================
# 6) Transform
# ============================================================

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# ============================================================
# 7) Robust image listing
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

bad_images = []

def is_readable_image(path: Path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception as e:
        bad_images.append({"image_path": str(path), "error": repr(e)})
        return False

def list_images_safe(folder: Path, recursive=True, verify_images=True):
    folder = Path(folder)

    if recursive:
        iterator = sorted(folder.rglob("*"))
    else:
        iterator = sorted(folder.glob("*"))

    paths = []

    for p in iterator:
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            if verify_images:
                if is_readable_image(p):
                    paths.append(p)
            else:
                paths.append(p)

    return paths

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]

        try:
            img = Image.open(p).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed to open image: {p} | Error: {repr(e)}")

        return self.transform(img), str(p)

class TestImageDataset(Dataset):
    def __init__(self, items, transform):
        self.items = list(items)
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        p, label = self.items[idx]

        try:
            img = Image.open(p).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed to open image: {p} | Error: {repr(e)}")

        return self.transform(img), int(label), str(p)

# ============================================================
# 8) Feature extractor: EfficientNet-B5 for PaDiM
# ============================================================

class EfficientNetPaDiM(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = efficientnet_b5(weights=EfficientNet_B5_Weights.DEFAULT)
        self.backbone.eval()

        for p in self.backbone.parameters():
            p.requires_grad = False

        self.features = []

        def hook(_, __, output):
            self.features.append(output)

        self.backbone.features[3].register_forward_hook(hook)
        self.backbone.features[5].register_forward_hook(hook)
        self.backbone.features[7].register_forward_hook(hook)

    @torch.no_grad()
    def forward(self, x):
        self.features = []

        _ = self.backbone(x)

        if len(self.features) != 3:
            raise RuntimeError(f"Expected 3 feature maps, got {len(self.features)}.")

        aligned = []

        for f in self.features:
            f = F.adaptive_avg_pool2d(f, (FEAT_GRID, FEAT_GRID))
            aligned.append(f)

        emb = torch.cat(aligned, dim=1)

        B, C, H, W = emb.shape
        patches = emb.view(B, C, H * W).permute(0, 2, 1)

        return patches

# ============================================================
# 9) Scoring functions
# ============================================================

@torch.no_grad()
def image_score_from_patch_map(patch_scores_BHW):
    if SCORE_MODE == "max":
        return patch_scores_BHW.max(dim=1).values

    elif SCORE_MODE == "topk_mean":
        k = max(1, int(patch_scores_BHW.shape[1] * TOPK_FRAC))
        return torch.topk(patch_scores_BHW, k=k, largest=True).values.mean(dim=1)

    else:
        raise ValueError(f"Unknown SCORE_MODE: {SCORE_MODE}")

@torch.no_grad()
def fit_padim_statistics(model, loader, emb_indices):
    model.eval()

    sum_x = None
    sum_xx = None
    n_imgs = 0

    for xb, _ in tqdm(loader, desc="Fitting PaDiM statistics"):
        xb = xb.to(DEVICE, non_blocking=True)

        patches = model(xb)
        patches = patches[..., emb_indices.to(DEVICE)]
        patches = F.normalize(patches, p=2, dim=-1)

        B, HW, D = patches.shape

        if sum_x is None:
            sum_x = torch.zeros((HW, D), dtype=torch.float32, device="cpu")
            sum_xx = torch.zeros((HW, D, D), dtype=torch.float32, device="cpu")

        patches_cpu = patches.detach().cpu().float()

        sum_x += patches_cpu.sum(dim=0)
        sum_xx += torch.einsum("bhd,bhe->hde", patches_cpu, patches_cpu)

        n_imgs += B

        del xb, patches, patches_cpu

    if n_imgs < 2:
        raise ValueError("Need at least two training images.")

    mu = sum_x / n_imgs

    cov = (sum_xx / (n_imgs - 1)) - (n_imgs / (n_imgs - 1)) * torch.einsum(
        "hd,he->hde",
        mu,
        mu,
    )

    eye = torch.eye(cov.shape[-1], dtype=torch.float32).unsqueeze(0)
    cov = cov + COV_EPS * eye

    print("Inverting covariance matrices...")
    inv_cov = torch.linalg.inv(cov)

    return mu, inv_cov

@torch.no_grad()
def mahalanobis_scores(model, xb, mu_device, inv_cov_device, emb_indices):
    patches = model(xb)
    patches = patches[..., emb_indices.to(DEVICE)]
    patches = F.normalize(patches, p=2, dim=-1)

    diff = patches - mu_device.unsqueeze(0)

    m = torch.einsum(
        "bhd,hde,bhe->bh",
        diff,
        inv_cov_device,
        diff,
    )

    m = torch.clamp(m, min=0.0).sqrt()

    return m

# ============================================================
# 10) Load data
# ============================================================

print("\nScanning training images...")
train_paths = list_images_safe(train_good_path, recursive=True, verify_images=True)

if len(train_paths) == 0:
    raise ValueError("No valid training images found.")

print("Valid training normal images:", len(train_paths))

def get_label(folder_name):
    name = folder_name.lower().replace("_", "-").strip()

    if name == "non-defects":
        return 0

    if name == "defects":
        return 1

    return None

test_items = []

print("\nScanning test images...")

for folder in sorted(test_root_path.iterdir()):
    if not folder.is_dir():
        continue

    label = get_label(folder.name)

    if label is None:
        print("Skipping unknown folder:", folder.name)
        continue

    paths = list_images_safe(folder, recursive=True, verify_images=True)

    print(folder.name, "valid images:", len(paths))

    for p in paths:
        test_items.append((p, label))

if len(bad_images) > 0:
    bad_df = pd.DataFrame(bad_images)
    bad_df.to_csv(BAD_IMAGES_CSV, index=False)

    print("\nSkipped unreadable images:", len(bad_images))
    print("Saved bad image list:", BAD_IMAGES_CSV)

if len(test_items) == 0:
    raise ValueError("No valid test images found.")

normal_count = sum(1 for _, y in test_items if y == 0)
defect_count = sum(1 for _, y in test_items if y == 1)

print("\nTotal valid test images:", len(test_items))
print("Valid normal test images:", normal_count)
print("Valid defect test images:", defect_count)

if normal_count < 900:
    raise RuntimeError(
        f"Normal test count is too low: {normal_count}. "
        "Expected around 1038 for Lusitano. "
        "Your local dataset copy may be incomplete or folder structure may be wrong."
    )

if defect_count < 1500:
    raise RuntimeError(
        f"Defect test count is too low: {defect_count}. "
        "Expected around 1646 for Lusitano."
    )

train_loader = DataLoader(
    ImagePathDataset(train_paths, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

model = EfficientNetPaDiM().to(DEVICE).eval()
backbone_size_mb = model_size_mb(model)

with torch.no_grad():
    dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    C_total = model(dummy).shape[-1]
    del dummy

print("\nTotal embedding channels:", C_total)

if EMB_DIM > C_total:
    print(f"Warning: EMB_DIM={EMB_DIM} exceeds C_total={C_total}. Clamping.")
    EMB_DIM = C_total

rng = np.random.RandomState(SEED)
emb_indices = torch.from_numpy(
    rng.choice(C_total, size=EMB_DIM, replace=False)
).long()

print(f"Using embedding dims: {EMB_DIM} / {C_total}")

# ============================================================
# 11) Fit PaDiM statistics
# ============================================================

reset_peak_memory()

fit_start = time.time()

mu, inv_cov = fit_padim_statistics(model, train_loader, emb_indices)

fit_time_sec = time.time() - fit_start

print("\nPaDiM statistics ready")
print("mu shape:", tuple(mu.shape))
print("inv_cov shape:", tuple(inv_cov.shape))
print(f"PaDiM fit time sec: {fit_time_sec:.3f}")

padim_statistics_size_mb = tensor_size_mb(mu) + tensor_size_mb(inv_cov)
estimated_total_footprint_mb = backbone_size_mb + padim_statistics_size_mb

print(f"PaDiM statistics size MB: {padim_statistics_size_mb:.2f}")
print(f"Backbone/model size MB: {backbone_size_mb:.2f}")
print(f"Estimated total footprint MB: {estimated_total_footprint_mb:.2f}")

mu_device = mu.to(DEVICE, non_blocking=True)
inv_cov_device = inv_cov.to(DEVICE, non_blocking=True)

# ============================================================
# 12) Threshold
# ============================================================

rng_thresh = np.random.default_rng(SEED + 100)

num_for_thresh = min(THRESH_SAMPLE_IMAGES, len(train_paths))
thresh_indices = rng_thresh.permutation(len(train_paths))[:num_for_thresh]
thresh_subset = [train_paths[i] for i in thresh_indices]

thresh_loader = DataLoader(
    ImagePathDataset(thresh_subset, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

threshold_scores = []

print("\nComputing threshold from normal training subset...")

for xb, _ in tqdm(thresh_loader, desc="Threshold"):
    xb = xb.to(DEVICE, non_blocking=True)

    patch_scores = mahalanobis_scores(
        model,
        xb,
        mu_device,
        inv_cov_device,
        emb_indices,
    )

    img_scores = image_score_from_patch_map(patch_scores)

    threshold_scores.extend(img_scores.detach().cpu().numpy().tolist())

    del xb, patch_scores, img_scores

threshold_scores = np.array(threshold_scores)

threshold = threshold_scores.mean() + 3.0 * threshold_scores.std(ddof=1)

print(f"Internal threshold for F1: {threshold:.6f}")

# ============================================================
# 13) Test evaluation
# ============================================================

test_loader = DataLoader(
    TestImageDataset(test_items, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

reset_peak_memory()

torch.cuda.synchronize()

end_to_end_start = time.perf_counter()
compute_total_time = 0.0

y_true = []
y_score = []
image_paths = []

print("\nEvaluating test set...")

for xb, yb, paths in tqdm(test_loader, desc="Testing"):
    torch.cuda.synchronize()
    compute_start = time.perf_counter()

    xb = xb.to(DEVICE, non_blocking=True)

    patch_scores = mahalanobis_scores(
        model,
        xb,
        mu_device,
        inv_cov_device,
        emb_indices,
    )

    img_scores = image_score_from_patch_map(patch_scores)

    torch.cuda.synchronize()
    compute_total_time += time.perf_counter() - compute_start

    y_score.extend(img_scores.detach().cpu().numpy().tolist())
    y_true.extend([int(v) for v in yb])
    image_paths.extend([str(p) for p in paths])

    del xb, patch_scores, img_scores

torch.cuda.synchronize()

end_to_end_total_time = time.perf_counter() - end_to_end_start
compute_inference_time_per_image = compute_total_time / len(y_true)
end_to_end_local_runtime_per_image = end_to_end_total_time / len(y_true)

peak_memory_usage_mb = get_peak_memory_mb()

# ============================================================
# 14) Metrics
# ============================================================

y_true = np.array(y_true)
y_score = np.array(y_score)

y_pred = (y_score > threshold).astype(int)

auc_roc = roc_auc_score(y_true, y_score)
map_ap = average_precision_score(y_true, y_score)
f1 = f1_score(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

edge_eff = edge_efficiency_score(
    auc=auc_roc,
    ap=map_ap,
    f1=f1,
    time_per_image=compute_inference_time_per_image,
    total_mb=estimated_total_footprint_mb,
    peak_gpu_mb=peak_memory_usage_mb,
)

# ============================================================
# 15) Results
# ============================================================

result = {
    "Method": METHOD_NAME,
    "Seed": SEED,
    "Dataset_Root_Used": str(DATASET_ROOT),
    "IMG_SIZE": IMG_SIZE,
    "BATCH_SIZE": BATCH_SIZE,
    "NUM_WORKERS": NUM_WORKERS,
    "Score_Mode": SCORE_MODE,
    "TopK_Frac": TOPK_FRAC if TOPK_FRAC is not None else "N/A",
    "FEAT_GRID": FEAT_GRID,
    "EMB_DIM": EMB_DIM,
    "Internal_Threshold": threshold,
    "AUC_ROC": auc_roc,
    "mAP_AP": map_ap,
    "F1_Score": f1,
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp),
    "Inference_Time_Per_Image_sec": compute_inference_time_per_image,
    "Compute_Inference_Time_Per_Image_sec": compute_inference_time_per_image,
    "End_To_End_Local_Runtime_Per_Image_sec": end_to_end_local_runtime_per_image,
    "Compute_Total_Time_sec": compute_total_time,
    "End_To_End_Local_Total_Time_sec": end_to_end_total_time,
    "Memory_Bank_Size_MB": 0.0,
    "PaDiM_Statistics_Size_MB": padim_statistics_size_mb,
    "Backbone_Model_Size_MB": backbone_size_mb,
    "Estimated_Total_Footprint_MB": estimated_total_footprint_mb,
    "Peak_Memory_Usage_MB": peak_memory_usage_mb,
    "Edge_Efficiency": edge_eff,
    "Train_Normal_Images": len(train_paths),
    "Test_Normal_Images": normal_count,
    "Test_Defect_Images": defect_count,
    "Total_Test_Images": len(test_items),
    "Skipped_Bad_Images": len(bad_images),
    "PaDiM_Fit_Time_sec": fit_time_sec,
}

df_result = pd.DataFrame([result])

df_scores = pd.DataFrame({
    "image_path": image_paths,
    "gt_label": y_true,
    "padim_score": y_score,
    "pred_label": y_pred,
})

df_result.to_csv(RESULT_CSV, index=False)
df_scores.to_csv(SCORES_CSV, index=False)

print("\n==============================")
print("FIXED PADIM MAX PATCH RESULT")
print("==============================")
display(df_result)

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Normal", "Defect"],
    digits=4,
))

print("\nSaved result:")
print(RESULT_CSV)

print("\nSaved image scores:")
print(SCORES_CSV)

if len(bad_images) > 0:
    print("\nSaved skipped bad images:")
    print(BAD_IMAGES_CSV)